In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
from dash import Dash, html, dcc, Input, Output
import dash_bootstrap_components as dbc
from dash_bootstrap_templates import load_figure_template

In [26]:
df = pd.read_excel('saida_controlados_paciente.xlsx')

In [27]:
df_temp = df.sample(frac=1, random_state=42).reset_index(drop=True)
codigos_unicos, _ = pd.factorize(df_temp['Cod_Pac'])
df_temp['Paciente_ID'] = [f"PAC_{i+1:04d}" for i in codigos_unicos]
df_analitico = df_temp.drop(columns=['Cod_Pac'])

In [28]:
df_analitico.head()

,Origem,Cod_Produto,Medicamento,Quantidade,Idade,Gênero,ESF,Val. Médio R$,Val. Total. R$,Paciente_ID
0,FARMACIA MUNICIPAL - FARMACIA,685.0,FLUOXETINA CÁPSULA 20 MG,280.0,NaN,NaN,NaN,0.069,19.32,PAC_0001
1,FARMACIA MUNICIPAL - FARMACIA,154.0,IMIPRAMINA CP 25 MG,720.0,NaN,NaN,NaN,0.430,309.60,PAC_0002
2,FARMACIA MUNICIPAL - FARMACIA,1017.0,CODEINA CP 30MG,24.0,41.0,M,AGUAPES,1.020,24.48,PAC_0003
3,FARMACIA MUNICIPAL - FARMACIA,1017.0,CODEINA CP 30MG,12.0,31.0,M,NaN,1.020,12.24,PAC_0004
4,FARMACIA MUNICIPAL - FARMACIA,1017.0,CODEINA CP 30MG,20.0,NaN,NaN,NaN,1.020,20.40,PAC_0005


In [29]:
df_analitico.drop(columns=['Origem'], inplace=True)

In [30]:
condicoes = [
    df_analitico['Medicamento'].str.contains('AMITRIPTILINA|IMIPRAMINA|FLUOXETINA|PAROXETINA|NORTRIPTILINA', case=False, na=False),
    df_analitico['Medicamento'].str.contains('BROMAZEPAM|ALPRAZOLAM|CLONAZEPAM|DIAZEPAM', case=False, na=False),
    df_analitico['Medicamento'].str.contains('CODEINA|MORFINA|TRAMADOL', case=False, na=False),
    df_analitico['Medicamento'].str.contains('FENOBARBITAL|FENITOINA', case=False, na=False),
    df_analitico['Medicamento'].str.contains('CARBAMAZEPINA|ACIDO VALPROICO|VALPROATO DE SODIO', case=False, na=False),
    df_analitico['Medicamento'].str.contains('CARBONATO DE LITIO', case=False, na=False),
    df_analitico['Medicamento'].str.contains('BIPERIDENO', case=False, na=False),
    df_analitico['Medicamento'].str.contains('RISPERIDONA|CLORPROMAZINA|LEVOMEPROMAZINA|HALOPERIDOL|SULPIRIDA|PERICIAZINA|TIORIDAZINA', case=False, na=False)
]

In [31]:
classes = [
    'Antidepressivo',
    'Ansiolítico (Benzodiazepínico)',
    'Analgésico',
    'Anticonvulsivante',
    'Estabilizador humor/anticonvulsivante',
    'Estabilizador humor',
    'Antiparkinsoniano',
    'Neuroléptico'
]

In [32]:
df_analitico['Classe_Terapeutica'] = np.select(condicoes, classes, default='Outros')

In [33]:
genero_contagem = df['Gênero'].value_counts().reset_index()
esf = df_analitico['ESF'].value_counts()

In [34]:
top_10 = df_analitico.groupby('Medicamento')['Quantidade'].sum().sort_values(ascending=False).head(10)
top_10_df = top_10.reset_index()

In [37]:
top_10_freq = df_analitico['Medicamento'].value_counts().head(10)
top_10_freq_df = top_10_freq.reset_index()


In [38]:
top_classes = df_analitico['Classe_Terapeutica'].value_counts()
top_classes_df = top_classes.reset_index()

In [39]:
top_classes_quant = df_analitico.groupby('Classe_Terapeutica')['Quantidade'].sum().sort_values(ascending=False)
top_classes_quant_reset = top_classes_quant.reset_index()

In [40]:
resumo_custo_classe = (
    df_analitico.groupby('Classe_Terapeutica')['Val. Total. R$']
    .sum()
    .reset_index(name='Custo_Total')
    .sort_values(by='Custo_Total', ascending=False) # Ordem crescente para o maior ficar no topo
)

In [41]:
perfil_pacientes = df_analitico.groupby('Paciente_ID').agg(Total_Med_Distintos = ('Medicamento','nunique'), Total_Classes_Distintas = ('Classe_Terapeutica', 'nunique'), Total_Retiradas = ('Medicamento', 'count')).reset_index()
total_pacientes_geral = len(perfil_pacientes)
pacientes_multiplos_meds = perfil_pacientes[perfil_pacientes['Total_Med_Distintos'] > 1]
pacientes_multiplos_classes = perfil_pacientes[perfil_pacientes['Total_Classes_Distintas'] > 1]
perc_multiplos_meds = (len(pacientes_multiplos_meds)/total_pacientes_geral)*100
perc_multiplos_classes = (len(pacientes_multiplos_classes)/ total_pacientes_geral)*100

In [42]:
dados_grafico = pd.DataFrame({
    'Indicador': [
        'Múltiplos Medicamentos', 
        'Múltiplas Classes Terapêuticas'
    ],
    'Percentual': [
        perc_multiplos_meds, 
        perc_multiplos_classes
    ],
    'Quantidade_Pacientes': [
        len(pacientes_multiplos_meds), 
        len(pacientes_multiplos_classes)
    ]
})

In [43]:
tabela_esf_med = pd.crosstab(df_analitico['ESF'], df_analitico['Classe_Terapeutica'])
tabela_esf_med_longo = tabela_esf_med.stack().reset_index(name='Quantidade')

In [44]:
#Relação entre valor gasto e ESF
resumo_financeiro_esf = df_analitico.groupby('ESF')['Val. Total. R$'].agg(
    Custo_Total=('sum'),
    Custo_Medio=('mean'),
    Total_Atendimentos=('count')
).reset_index()
resumo_financeiro_esf = resumo_financeiro_esf.sort_values(by='Custo_Total', ascending=False)

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [45]:
# 1. Agrupar os dados por ESF para criar o perfil estatístico de cada unidade
perfil_esf = df_analitico.groupby('ESF').agg(
    Idade_Media=('Idade', 'mean'),
    Volume_Dispensacoes=('Medicamento', 'count'),
    Custo_Total=('Val. Total. R$', 'sum'),
    Custo_Medio_Por_Atendimento=('Val. Total. R$', 'mean')
).reset_index()
# 2. Selecionar as variáveis numéricas que servirão de base para o agrupamento
features = ['Idade_Media', 'Volume_Dispensacoes', 'Custo_Total', 'Custo_Medio_Por_Atendimento']
X = perfil_esf[features]
# 3. Padronizar as variáveis (normalização z-score) para que escalas diferentes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# 4. Aplicar o K-Means (definindo 3 clusters/perfis diferentes)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
perfil_esf['Cluster'] = kmeans.fit_predict(X_scaled)


In [46]:
load_figure_template('zephyr')

app = Dash(__name__, external_stylesheets=[dbc.themes.ZEPHYR])

# --- FUNÇÃO PARA GRÁFICOS DINÂMICOS ---
def gerar_graficos_filtrados(dados, filtro_nome='Todos'):
    # 1. Distribuição do gênero
    genero_cont = dados['Gênero'].value_counts().reset_index()
    genero_cont.columns = ['Gênero', 'count']
    fig1 = px.pie(
        genero_cont, values='count', names='Gênero',
        title=f'Distribuição de gênero ({filtro_nome})',
        hole=0.3
    )
    fig1.update_layout(title={'x': 0.45})
    fig1.update_traces(sort=False)

    # 2. Distribuição das idades
    fig2 = px.histogram(
        dados, x='Idade', nbins=20,
        title=f'Distribuição da Idade ({filtro_nome})',
    )
    fig2.update_layout(title={'x': 0.45})
    fig2.update_traces(marker_line_color='black', marker_line_width=1.5)

    # 3. Distribuição por ESF
    esf_cont = dados['ESF'].value_counts()
    fig3 = px.bar(
        x=esf_cont.index, y=esf_cont.values,
        labels={'x': 'ESF', 'y': 'Frequência'},
        title=f'Distribuição das ESF por frequência ({filtro_nome})',
    )
    fig3.update_layout(title={'x': 0.45})

    # 4. Distribuição de Idade por ESF
    fig4 = px.box(
        dados, x='ESF', y='Idade',
        title=f'Distribuição de idade dos pacientes por ESF ({filtro_nome})'
    )
    fig4.update_layout(title={'x': 0.45})

    return fig1, fig2, fig3, fig4


# --- GRÁFICOS INICIAIS (DINÂMICOS) ---
fig1_initial, fig2_initial, fig3_initial, fig4_initial = gerar_graficos_filtrados(df, 'Todos')

# --- GRÁFICOS ESTÁTICOS (NÃO RESPONDEM AO FILTRO) ---
fig5 = px.bar(top_10_df, x='Quantidade', y='Medicamento', orientation='h', title='Top 10 medicamentos mais dispensados por quantidade', labels={'Medicamento': ''}, text='Quantidade', color='Quantidade')
fig5.update_layout(title={'x': 0.45})

fig6 = px.bar(top_10_freq_df, x='count', y='Medicamento', labels={'count': 'Frequência', 'Medicamento': 'Medicamento'}, title='Top 10 medicamentos mais dispensados por frequência', orientation='h')
fig6.update_layout(title={'x': 0.45})

fig7 = px.bar(top_classes_df, x='count', y='Classe_Terapeutica', orientation='h', labels={'count': 'Frequência', 'Classe_Terapeutica': 'Classe Terapêutica'}, title='Top classes terapêuticas mais dispensadas')
fig7.update_layout(title={'x': 0.45})

fig8 = px.bar(top_classes_quant_reset, x='Quantidade', y='Classe_Terapeutica', orientation='h', title='Top classes terapêuticas mais dispensadas por quantidade', labels={'Classe_Terapeutica': ''}, text='Quantidade', color='Quantidade')
fig8.update_layout(title={'x': 0.45})

fig9 = px.bar(resumo_custo_classe, x='Custo_Total', y='Classe_Terapeutica', orientation='h', title='Custo total por Classe Terapêutica', labels={'Classe_Terapeutica': ''}, text='Custo_Total', color='Custo_Total')
fig9.update_layout(title={'x': 0.45})

fig10 = px.bar(dados_grafico, x='Indicador', y='Percentual', text=dados_grafico['Percentual'].round(2), title='Perfil de Polimedicação dos Pacientes', labels={'Indicador': 'Categoria', 'Percentual': 'Participação %'}, color='Indicador')
fig10.update_layout(title={'x': 0.45})

fig11 = px.histogram(df_analitico, x='Quantidade', y='Classe_Terapeutica', color='Gênero', title='Relação entre a classe terapêutica e gênero')
fig11.update_layout(title={'x': 0.45})

fig12 = px.box(df_analitico, x='Idade', y='Classe_Terapeutica', title='Relação entre a classe terapêutica e idade dos pacientes', width=700, height=400)
fig12.update_layout(title={'x':0.45}) 

fig13 = px.bar(tabela_esf_med_longo, x='ESF', y='Quantidade', color='Classe_Terapeutica', title='Distribuição das classes terapêuticas mais frequentes por ESF')
fig13.update_layout(title={'x': 0.45})

fig14 = px.box(df_analitico, x='Gênero', y='Val. Total. R$', color='Gênero', title='Distribuição do Valor Total Dispensado por Gênero', points='outliers')
fig14.update_layout(title={'x': 0.45})

fig15 = px.bar(resumo_financeiro_esf, x='Custo_Total', y='ESF', orientation='h', title='Custo Total por Unidade de Saúde (ESF)')
fig15.update_layout(title={'x': 0.45})

fig16 = px.scatter(perfil_esf, x='Volume_Dispensacoes', y='Idade_Media', color='Cluster', text='ESF', size='Custo_Total', title='Clusterização das Unidades de Saúde (ESF) por Perfil de Atendimento', labels={'Volume_Dispensacoes': 'Volume Total de Dispensações', 'Idade_Media': 'Idade Média dos Pacientes (Anos)'})
fig16.update_traces(textposition = 'top center')
fig16.update_layout(title={'x':0.45}, height=500)

# --- OPÇÕES DO DROPDOWN ---
opcoes_dropdown = [{'label': 'Todas as ESFs', 'value': 'Todos'}] + [
    {'label': u, 'value': u} for u in sorted(df['ESF'].dropna().unique())
]

# --- LAYOUT ---
app.layout = dbc.Container([
    dbc.Row(dbc.Col(
        html.H4("Perfil do uso de medicamentos psicotrópicos por usuários da Farmácia Municipal de Osório - RS", className="bg-primary text-white p-3 rounded mt-2"),
        width=12
    )),

    dbc.Row(dbc.Col(html.H5("Perfil sociodemográfico dos pacientes usuários de medicamentos psicotrópicos", className="text-primary border-bottom pb-2 mt-3"), width={'size': 10, 'offset': 1})),

    dbc.Row([
        dbc.Col([
            html.Label("Filtrar por Unidade de Saúde (ESF):", className="fw-bold mt-2"),
            dcc.Dropdown(
                id='dropdown-esf',
                options=opcoes_dropdown,
                value='Todos',
                clearable=False
            )
        ], width={'size': 4, 'offset': 1}, className="mb-4")
    ]),

    # Gráficos que respondem ao filtro
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoPizza', figure=fig1_initial), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id='graficoHist', figure=fig2_initial), width={'size': 5}),
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoBarra1', figure=fig3_initial), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id='graficoBox1', figure=fig4_initial), width={'size': 5})
    ]),

    # Gráficos estáticos
    dbc.Row(dbc.Col(html.H5("Perfil de dispensação de medicamentos psicotrópicos", className="text-primary border-bottom pb-2 mt-4"), width={'size': 10, 'offset': 1})),
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoBarra2', figure=fig5), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id='graficoBarra3', figure=fig6), width={'size': 5})   
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoBarra4', figure=fig7), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id='graficoBarra5', figure=fig8), width={'size': 5})
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoBarra6', figure=fig9), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id='graficoBarra7', figure=fig10), width={'size': 5})
    ]),

    dbc.Row(dbc.Col(html.H5("Perfil de custos, gênero e classe terapêutica", className="text-primary border-bottom pb-2 mt-4"), width={'size': 10, 'offset': 1})),
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoHist2', figure=fig11), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id= 'graficoBox2', figure=fig12), width={'size':5, 'offset':0}),
        dbc.Col(dcc.Graph(id='graficoBarra8', figure=fig13), width={'size': 5, 'offset':3})
    ]),
    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoBox2', figure=fig14), width={'size': 5, 'offset': 1}),
        dbc.Col(dcc.Graph(id='graficoBarra9', figure= fig15), width={'size':5, 'offset':0})   
    ], className="mb-5"),

    dbc.Row(dbc.Col(
        html.H5("Segmentação das Unidades de Saúde (Clusterização K-Means)", className="text-primary border-bottom pb-2 mt-4"),
        width={'size': 10, 'offset': 1}
    )),

    dbc.Row([
        dbc.Col(dcc.Graph(id='graficoDispersao', figure=fig16), width={'size':6, 'offset': 3})
    ], className='mb-5')

], fluid=True)

# --- CALLBACK (PARA O FILTRO) ---
@app.callback(
    Output('graficoPizza', 'figure'),
    Output('graficoHist', 'figure'),
    Output('graficoBarra1', 'figure'),
    Output('graficoBox1', 'figure'),
    Input('dropdown-esf', 'value'),
    prevent_initial_call=True
)
def update_dropdown(value):
    df_filtrado = df if value == 'Todos' else df[df['ESF'] == value]
    return gerar_graficos_filtrados(df_filtrado, value)

if __name__ == '__main__':
    app.run(debug=True, port=8051)

[2026-11-25 18:06:52,149] ERROR in app: Exception on /_alive_ff1bff2f-5b8f-4ea8-82ce-1636b6dd2e8e [GET]
Traceback (most recent call last):
  File "C:\Users\nandi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\flask\app.py", line 878, in full_dispatch_request
    rv = self.preprocess_request()
  File "C:\Users\nandi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\flask\app.py", line 1253, in preprocess_request
    rv = self.ensure_sync(before_func)()
  File "C:\Users\nandi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\dash\dash.py", line 1449, in _setup_server
    _validate.validate_layout(self.layout, self._layout_value())
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\nandi\AppData\Local\Packages\PythonSoftwareFoundation.Py